# 09. От разложения Эвальда к PME

**Практикум к слайдам 15–17 второй лекции.** Основная часть: 45–60 минут с обсуждением. Все системы создаются кодом; GPU и внешние структуры не нужны.

**Цели:** увидеть, как одно взаимодействие делится на две части; понять роль гауссовой функции и self correction; проверить независимость полной энергии от α; исследовать сходимость и сравнить энергию и силы с OpenMM PME.

**Запуск в Google Colab:** загрузите `.ipynb` через «Файл → Загрузить блокнот», оставьте CPU и выполните все ячейки по порядку. Первая ячейка устанавливает зависимости. Установка и графики могут занять несколько минут; расчёты маленькой системы обычно гораздо короче. Для изменения параметров повторяйте зависимые ячейки. Сохранённые результаты соответствуют исходным параметрам.

Работаем с **фиксированными зарядами**, а не траекторией. Нет LJ, связей, исключений пар, термостата и интегрирования движения. Не запускайте MD этой модели с противоположными точечными зарядами без короткодействующего отталкивания.

Единицы: nm, заряд в e, энергия в kJ/mol, сила в kJ/mol/nm. Кубическая нейтральная ячейка. Внешние граничные условия — проводящие (*tin foil*), поверхностный дипольный член равен нулю. Это модель бесконечно периодической системы, а не изолированной группы зарядов в вакууме.

In [ ]:
%pip -q install openmm==8.3.1 numpy scipy matplotlib ipywidgets


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf, erfc
from itertools import product
from time import perf_counter
import openmm as mm
from openmm import unit
from IPython.display import display
plt.rcParams.update({'figure.figsize': (9,4), 'axes.grid': True, 'font.size': 11})
C = 138.93545764438198  # 1/(4π ε0) в kJ mol^-1 nm e^-2; константа OpenMM
backend = mm.Platform.getPlatformByName('Reference')
print('OpenMM', mm.__version__, '| платформа', backend.getName())


## 1. Что именно повторяется в пространстве?

Создаём восемь зарядов: четыре положительных и четыре отрицательных. Небольшое случайное смещение нарушает идеальную симметрию кристалла. Каждый заряд взаимодействует с остальными зарядами и периодическими образами. Собственное взаимодействие с тем же зарядом в **той же** ячейке исключено, с его образами в других ячейках — учитывается.

На рисунке показаны центральная ячейка и 26 ближайших копий. Это иллюстрация периодичности, а не полный набор зарядов для расчёта энергии.

In [ ]:
L = 2.4
rng = np.random.default_rng(2026)
pos = np.array(list(product([0.65,1.75], repeat=3))) + rng.uniform(-0.16,0.16,(8,3))
q = np.array([1,-1,-1,1,-1,1,1,-1.], dtype=float)
assert abs(q.sum()) < 1e-12
colors = np.where(q>0, 'tab:red', 'tab:blue')
d = pos[:,None,:]-pos[None,:,:]; d -= L*np.round(d/L)
r = np.linalg.norm(d,axis=-1); np.fill_diagonal(r,np.inf)
print('N =',len(q),'Q =',q.sum(),'e; минимальное расстояние =',r.min(),'nm')
fig=plt.figure(figsize=(12,5)); ax=fig.add_subplot(121,projection='3d'); bx=fig.add_subplot(122,projection='3d')
ax.scatter(*pos.T,c=colors,s=80)
for i,p in enumerate(pos): ax.text(*p,str(i))
for shift in product([-1,0,1],repeat=3):
    p=pos+L*np.array(shift)
    bx.scatter(*p.T,c=colors,s=25 if shift==(0,0,0) else 8,alpha=1 if shift==(0,0,0) else .18)
for a in [ax,bx]:a.set(xlabel='x, nm',ylabel='y, nm',zlabel='z, nm');a.set_box_aspect((1,1,1))
ax.set(xlim=(0,L),ylim=(0,L),zlim=(0,L),title='Центральная ячейка: номера зарядов')
bx.set_title('Центральная ячейка и 26 копий');plt.tight_layout();plt.show()


## 2. Одно взаимодействие — две согласованные части

$$\frac1r=\frac{\operatorname{erfc}(\alpha r)}r+\frac{\operatorname{erf}(\alpha r)}r.$$

Здесь α имеет размерность nm⁻¹. Первый член быстро затухает и подходит для прямой суммы; второй гладок в начале координат и удобен для расчёта в обратном пространстве. При $r\to0$ второй член стремится к $2\alpha/\sqrt\pi$, но при больших расстояниях сохраняет кулоновский хвост.

Вспомогательное облако единичного заряда:

$$\rho_G(\mathbf r)=\frac{\alpha^3}{\pi^{3/2}}e^{-\alpha^2r^2},\qquad \int\rho_G\,d^3r=1.$$

При большем α облако **уже**, а короткодействующая часть короче. Это не электронная плотность реального атома. На правом графике показана одномерная линия через трёхмерное облако: площадь под этой линией не обязана равняться единице.

In [ ]:
def plot_split(alpha=2.2):
    r=np.linspace(.025,2.4,500); x=np.linspace(-2,2,500)
    short=erfc(alpha*r)/r; smooth=erf(alpha*r)/r
    assert np.allclose(short+smooth,1/r)
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(r,1/r,'k--',label='1/r')
    ax[0].plot(r,short,label='erfc(αr)/r')
    ax[0].plot(r,smooth,label='erf(αr)/r')
    ax[0].set(xlabel='r, nm',ylabel='Ядро, nm⁻¹',ylim=(0,10),title=f'α={alpha:.2f} nm⁻¹');ax[0].legend()
    ax[1].plot(x,alpha**3/np.pi**1.5*np.exp(-alpha**2*x*x))
    ax[1].set(xlabel='x, nm',ylabel='ρG(x,0,0), nm⁻³',title='Линия через гауссово облако')
    plt.tight_layout();plt.show()
plot_split(2.2)


**Необязательный интерактивный просмотр.** Ползунок меняет α; исходный потенциал остаётся прежним. Если виджет не отображается, вызывайте `plot_split(1.2)` и `plot_split(3.2)` вручную.

In [ ]:
import ipywidgets as widgets
alpha_slider = widgets.FloatSlider(value=2.2,min=.6,max=4.,step=.2,
                                  description='α, nm⁻¹',continuous_update=False)
def redraw_split(change):
    with split_output:
        split_output.clear_output(wait=True)
        plot_split(change['new'])
split_output=widgets.Output()
alpha_slider.observe(redraw_split,names='value')
display(alpha_slider,split_output)


## 3. Полная энергия: прямая сумма, обратная сумма и self correction

Пусть $\mathbf r_{ij\mathbf n}=\mathbf r_i-\mathbf r_j+L\mathbf n$, $V=L^3$, $\mathbf k=2\pi\mathbf m/L$ и

$$S(\mathbf k)=\sum_jq_j e^{i\mathbf k\cdot\mathbf r_j}.$$

В выбранных единицах обозначим кулоновскую константу через C:

$$U_{\rm real}=\frac C2\sum_{ij}\sum_{\mathbf n}'q_iq_j\frac{\operatorname{erfc}(\alpha r_{ij\mathbf n})}{r_{ij\mathbf n}},$$

$$U_{\rm rec}=\frac{2\pi C}{V}\sum_{\mathbf k\ne0}\frac{e^{-k^2/(4\alpha^2)}}{k^2}|S(\mathbf k)|^2,$$

$$U_{\rm self}=-\frac{C\alpha}{\sqrt\pi}\sum_iq_i^2,\qquad U=U_{\rm real}+U_{\rm rec}+U_{\rm self}.$$

Штрих убирает только $i=j,\mathbf n=0$. Собственная поправка убирает искусственный вклад взаимодействия заряда с его вспомогательным облаком. Она не зависит от координат и потому не даёт силы. Нулевая мода исключена; код намеренно требует нейтральности, не вводя компенсирующего фона.

В конечном расчёте используем сферическое отсечение $r<r_{\rm cut}$ в прямой сумме и куб индексов $-n_{\max}\le m_x,m_y,m_z\le n_{\max}$ в обратной. Это учебный прямой перебор, не эффективная реализация для больших систем. В прямой сумме допустим радиус больше L/2, потому что явно перечисляются все нужные периодические образы. В PME OpenMM используется другая организация прямой части и радиус меньше L/2.

### Аналитические силы

Силы вычисляются как $-\nabla_iU$. Для прямой части коэффициент при векторе $\mathbf r_{ij\mathbf n}$ равен

$$Cq_iq_j\left[\frac{\operatorname{erfc}(\alpha r)}{r^3}+\frac{2\alpha}{\sqrt\pi}\frac{e^{-\alpha^2r^2}}{r^2}\right].$$

Для обратной части при указанном знаке экспоненты:

$$\mathbf F_{i,\rm rec}=\frac{4\pi C q_i}{V}\sum_{\mathbf k\ne0}\frac{e^{-k^2/(4\alpha^2)}}{k^2}\operatorname{Im}\left[e^{i\mathbf k\cdot\mathbf r_i}S(\mathbf k)^*\right]\mathbf k.$$

Множители 1/2 и знаки легко перепутать: ниже предусмотрены независимые проверки.

In [ ]:
def ewald(positions,charges,box,alpha,rcut,nmax):
    p=np.asarray(positions)%box; charges=np.asarray(charges)
    if abs(charges.sum())>1e-10:raise ValueError('Этот пример требует нейтральной ячейки.')
    shifts=np.array(list(product(range(-int(np.ceil(rcut/box)),int(np.ceil(rcut/box))+1),repeat=3)))*box
    d=p[:,None,None,:]-p[None,:,None,:]+shifts[None,None,:,:]
    r=np.linalg.norm(d,axis=-1)
    keep=(r>1e-12)&(r<rcut);safe=np.where(keep,r,1.)
    qq=charges[:,None,None]*charges[None,:,None]
    screened=np.where(keep,erfc(alpha*safe)/safe,0.)
    real=.5*C*np.sum(qq*screened)
    coeff=np.where(keep,erfc(alpha*safe)/safe**3+2*alpha/np.sqrt(np.pi)*np.exp(-(alpha*safe)**2)/safe**2,0.)
    fr=C*np.sum(qq[...,None]*coeff[...,None]*d,axis=(1,2))
    ns=np.array(list(product(range(-nmax,nmax+1),repeat=3)),float)
    ns=ns[np.any(ns!=0,axis=1)];kv=2*np.pi/box*ns;k2=np.sum(kv*kv,axis=1)
    phase=np.exp(1j*(p@kv.T));S=charges@phase
    w=np.exp(-k2/(4*alpha**2))/k2
    reciprocal=C*2*np.pi/box**3*np.sum(w*np.abs(S)**2)
    fk=C*4*np.pi/box**3*charges[:,None]*((np.imag(phase*np.conj(S)[None,:])*w[None,:])@kv)
    own=-C*alpha/np.sqrt(np.pi)*np.sum(charges**2)
    return np.array([real,reciprocal,own]),fr+fk


### Проверки реализации и выбор эталона

Сначала увеличиваем границы обеих сумм. Затем проверяем силу конечной разностью энергии и отсутствие суммарной внутренней силы. Разность двух больших расчётов — проверка сходимости для этой конфигурации, а не универсальная гарантия точности.

In [ ]:
ref_alpha, ref_rc, ref_n = 2.2, 7.2, 14
ref_terms,F_ref=ewald(pos,q,L,ref_alpha,ref_rc,ref_n)
E_ref=ref_terms.sum()
more_terms,F_more=ewald(pos,q,L,ref_alpha,8.4,16)
print('Вклады [real, reciprocal, self]:',ref_terms)
print('Эталонная энергия:',E_ref,'kJ/mol')
print('Изменение при расширении сумм:',more_terms.sum()-E_ref,'kJ/mol')
assert abs(more_terms.sum()-E_ref)<1e-7
assert np.max(np.abs(F_more-F_ref))<1e-7
h=1e-5; pp=pos.copy();pm=pos.copy();pp[0,0]+=h;pm[0,0]-=h
fd=-(ewald(pp,q,L,ref_alpha,ref_rc,ref_n)[0].sum()-ewald(pm,q,L,ref_alpha,ref_rc,ref_n)[0].sum())/(2*h)
print('Fx аналитическая:',F_ref[0,0],'; конечная разность:',fd)
assert np.isclose(fd,F_ref[0,0],rtol=1e-6,atol=1e-6)
assert np.linalg.norm(F_ref.sum(axis=0))<1e-8
fig=plt.figure(figsize=(6,5));ax=fig.add_subplot(111,projection='3d')
ax.scatter(*pos.T,c=colors,s=70)
scale=.25/np.max(np.linalg.norm(F_ref,axis=1))
ax.quiver(*pos.T,*(F_ref*scale).T,color='black',arrow_length_ratio=.2)
ax.set(xlabel='x, nm',ylabel='y, nm',zlabel='z, nm',title='Силы Эвальда: единый масштаб стрелок');plt.show()
print('Проверки пройдены.')


## 4. α меняет части, но не полную энергию

Меняем α при достаточно широких границах сумм. Левая панель показывает компенсацию вкладов. Средняя — малую остаточную ошибку суммы. Правая показывает неверную энергию без self correction. В точном разложении α не является физическим параметром системы.

При слишком малом или слишком большом α ранее выбранных границ может перестать хватать; тогда зависимость суммы от α сигнализирует о численной ошибке.

In [ ]:
alphas=np.linspace(1.2,3.2,9)
components=np.array([ewald(pos,q,L,a,ref_rc,ref_n)[0] for a in alphas])
fig,ax=plt.subplots(1,3,figsize=(15,4))
for j,name in enumerate(['real','reciprocal','self']):ax[0].plot(alphas,components[:,j],'o-',label=name)
ax[0].plot(alphas,components.sum(axis=1),'k-',label='Сумма');ax[0].legend();ax[0].set_ylabel('U, kJ/mol')
ax[1].plot(alphas,components.sum(axis=1)-E_ref,'o-');ax[1].set_ylabel('U − Uref, kJ/mol')
ax[2].plot(alphas,components[:,:2].sum(axis=1),'o-',label='Без self correction')
ax[2].axhline(E_ref,color='black',ls='--',label='Правильно');ax[2].legend();ax[2].set_ylabel('U, kJ/mol')
for a in ax:a.set_xlabel('α, nm⁻¹')
plt.tight_layout();plt.show()
assert np.max(np.abs(components.sum(axis=1)-E_ref))<1e-6


## 5. Почему одной степени 1/k² недостаточно?

Вес обратной суммы содержит **два** множителя:

$$w(k)=\frac{e^{-k^2/(4\alpha^2)}}{k^2}.$$

В трёх измерениях число мод в сферической оболочке приблизительно растёт как $k^2\,dk$. Поэтому одно уменьшение веса каждой моды как $1/k^2$ ещё не означает быстрого уменьшения суммы по оболочке. Кроме того, энергетический вклад содержит $|S(\mathbf k)|^2$. Гауссов множитель обеспечивает быстрое подавление высоких частот.

Слева показываем безразмерный вес при $u=k/\alpha$. Справа — реальные **положительные вклады энергии** по сферическим оболочкам для нашей системы. Полосы дискретны из-за решётки волновых векторов.

In [ ]:
u=np.linspace(.2,10,400)
ns=np.array(list(product(range(-10,11),repeat=3)),float);ns=ns[np.any(ns!=0,axis=1)]
kv=2*np.pi/L*ns;k2=np.sum(kv**2,axis=1);kn=np.sqrt(k2)
S=q@np.exp(1j*(pos@kv.T));bins=np.arange(0,kn.max()+3,3)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].semilogy(u,1/u**2,label='1/u²');ax[0].semilogy(u,np.exp(-u*u/4)/u**2,label='exp(−u²/4)/u²')
ax[0].set(xlabel='u = k/α',ylabel='Безразмерный вес');ax[0].legend()
for a in [1.2,2.2,3.2]:
    terms=C*2*np.pi/L**3*np.exp(-k2/(4*a*a))/k2*np.abs(S)**2
    sums,_=np.histogram(kn,bins=bins,weights=terms)
    # binned sum through cumulative weights can lose tiny tail increments;
    # compute each shell independently for an accurate logarithmic plot.
    sums=np.array([terms[(kn>=lo)&(kn<hi)].sum() for lo,hi in zip(bins[:-1],bins[1:])])
    centers=(bins[:-1]+bins[1:])/2;mask=sums>0
    ax[1].semilogy(centers[mask],sums[mask],'o-',label=f'α={a}')
ax[1].set(xlabel='|k|, nm⁻¹',ylabel='Вклад оболочки в Urec, kJ/mol',ylim=(1e-12,1e3));ax[1].legend()
plt.tight_layout();plt.show()


## 5а. Как сходятся прямая и фурье-суммы по отдельности?

В предыдущем разделе рассматривались вклады отдельных фурье-оболочек. Здесь строим **накопленные суммы** и непосредственно измеряем, насколько они приблизились к своему пределу.

Для каждого α задаём два независимых эталона:

$$
U_{\mathrm{real,ref}}(\alpha),\qquad U_{\mathrm{rec,ref}}(\alpha).
$$

Они вычисляются с большим радиусом прямой суммы и большим кубом фурье-индексов. Дополнительно увеличиваем обе границы и проверяем, что **каждый вклад** меняется меньше чем на $10^{-7}$ kJ/mol. Собственная поправка уже известна аналитически и не требует суммирования.

**Левые панели:** увеличиваем радиус $r_{\mathrm{cut}}$ при фиксированном α, включая всё больше реальных образов зарядов:

$$
\delta U_{\mathrm{real}}(r_{\mathrm{cut}};\alpha)
=\left|U_{\mathrm{real}}(r_{\mathrm{cut}};\alpha)-U_{\mathrm{real,ref}}(\alpha)\right|.
$$

**Правые панели:** увеличиваем максимальный индекс $n_{\max}$ куба в обратном пространстве:

$$
-n_{\max}\le m_x,m_y,m_z\le n_{\max},\qquad
\mathbf k=\frac{2\pi}{L}\mathbf m,\qquad\mathbf m\ne0,
$$

$$
\delta U_{\mathrm{rec}}(n_{\max};\alpha)
=\left|U_{\mathrm{rec}}(n_{\max};\alpha)-U_{\mathrm{rec,ref}}(\alpha)\right|.
$$

Здесь $n_{\max}$ — индекс куба, а не сферический радиус $k_{\max}$. Число ненулевых мод равно $(2n_{\max}+1)^3-1$.

Верхний ряд показывает сами частичные энергии; горизонтальный пунктир — их соответствующие пределы. Нижний ряд показывает абсолютные ошибки на логарифмической шкале. **Сравнивать каждый вклад с полной энергией или с пределом при другом α было бы неверно:** от α независима только правильно собранная полная энергия.

### Что ожидаем увидеть?

- При большем α ядро $\operatorname{erfc}(\alpha r)/r$ затухает быстрее: для прямой суммы достаточно меньшего радиуса.
- При большем α множитель $\exp[-k^2/(4\alpha^2)]$ затухает медленнее по k: для фурье-суммы требуется больше мод.
- Прямая энергия может сходиться немонотонно, поскольку произведения зарядов $q_iq_j$ имеют разные знаки. Случайная компенсация ошибок может дать отдельную особенно точную точку.
- Вклад каждой обратной моды в энергию неотрицателен. При вложенном расширении куба частичная фурье-энергия растёт к пределу, с точностью до округления. Это утверждение об энергии, а не о каждой компоненте силы.
- На уровне порядка $10^{-12}$ kJ/mol и ниже для этой задачи видны прежде всего ограничения машинной точности и вычитание почти одинаковых чисел. Для графика ошибки ограничены снизу этим уровнем.

**Задание:** найдите α, при котором прямая сумма сходится быстрее всего, и α, при котором быстрее всего сходится фурье-сумма. Почему эти значения различаются? Затем сопоставьте результат с картами общей ошибки в следующем разделе.


In [ ]:
# Каждый вклад сравниваем с эталоном при ТОМ ЖЕ alpha.
conv_alphas = [1.0, 2.0, 3.0]              # nm^-1
conv_rcuts = np.linspace(0.6, 6.0, 37)     # nm
conv_nmaxs = np.arange(1, 13)             # куб индексов -nmax...nmax
conv_ref_rcut, conv_ref_nmax = 8.4, 16
conv_floor = 1e-12                        # только нижний предел отображения
conv_results = {}

for conv_alpha in conv_alphas:
    conv_ref, _ = ewald(pos, q, L, conv_alpha, conv_ref_rcut, conv_ref_nmax)
    conv_check, _ = ewald(pos, q, L, conv_alpha, 9.6, 18)
    # Проверяем сходимость КАЖДОГО вклада, не только их суммы.
    assert np.max(np.abs(conv_check-conv_ref)) < 1e-7
    # Меняем только cutoff прямой суммы; обратная часть фиксирована.
    conv_real = np.array([
        ewald(pos, q, L, conv_alpha, float(r), conv_ref_nmax)[0][0]
        for r in conv_rcuts
    ])
    # Меняем только размер куба обратных индексов; прямая часть фиксирована.
    conv_recip = np.array([
        ewald(pos, q, L, conv_alpha, conv_ref_rcut, int(n))[0][1]
        for n in conv_nmaxs
    ])
    # Все слагаемые энергии обратной суммы неотрицательны.
    assert np.all(np.diff(conv_recip) >= -1e-9)
    assert np.all(conv_recip <= conv_ref[1]+1e-9)
    conv_results[conv_alpha] = (conv_ref, conv_real, conv_recip)

fig_conv, axes_conv = plt.subplots(2, 2, figsize=(13, 9), layout='constrained')
for conv_alpha, (conv_ref, conv_real, conv_recip) in conv_results.items():
    line, = axes_conv[0,0].plot(conv_rcuts, conv_real, '.-', label=f'α={conv_alpha:g}')
    color = line.get_color()
    axes_conv[0,0].axhline(conv_ref[0], color=color, ls='--', alpha=.55)
    axes_conv[0,1].plot(conv_nmaxs, conv_recip, 'o-', color=color, label=f'α={conv_alpha:g}')
    axes_conv[0,1].axhline(conv_ref[1], color=color, ls='--', alpha=.55)
    axes_conv[1,0].semilogy(conv_rcuts,
        np.maximum(np.abs(conv_real-conv_ref[0]), conv_floor), '.-', color=color)
    axes_conv[1,1].semilogy(conv_nmaxs,
        np.maximum(np.abs(conv_recip-conv_ref[1]), conv_floor), 'o-', color=color)

axes_conv[0,0].set(title='Прямая сумма: частичная энергия', ylabel='Ureal, kJ/mol')
axes_conv[0,1].set(title='Фурье-сумма: частичная энергия', ylabel='Urec, kJ/mol')
axes_conv[1,0].set(title='Ошибка прямой суммы', ylabel='|Ureal − Ureal,ref|, kJ/mol')
axes_conv[1,1].set(title='Ошибка фурье-суммы', ylabel='|Urec − Urec,ref|, kJ/mol')
for ax in axes_conv[:,0]: ax.set_xlabel('Радиус прямой суммы rcut, nm')
for ax in axes_conv[:,1]:
    ax.set_xlabel('Максимальный индекс nmax (куб в обратном пространстве)')
    ax.set_xticks([1,2,4,6,8,10,12])
for ax in axes_conv[0]: ax.legend()
for ax in axes_conv[1]:
    ax.axhline(1e-3, color='gray', ls=':', label='0.001 kJ/mol')
    ax.legend()
plt.show()

conv_target = 1e-3
print('Первые точки исследованной сетки с ошибкой ниже', conv_target, 'kJ/mol')
print('(первая удачная точка не гарантирует монотонности ошибки прямой суммы)')
for conv_alpha, (conv_ref, conv_real, conv_recip) in conv_results.items():
    real_ok = np.flatnonzero(np.abs(conv_real-conv_ref[0]) < conv_target)
    recip_ok = np.flatnonzero(np.abs(conv_recip-conv_ref[1]) < conv_target)
    real_text = f'{conv_rcuts[real_ok[0]]:.2f} nm' if len(real_ok) else 'не достигнуто'
    recip_text = str(conv_nmaxs[recip_ok[0]]) if len(recip_ok) else 'не достигнуто'
    print(f'α={conv_alpha:g}: rcut={real_text}; nmax={recip_text}')
print('Пунктир сверху: собственный сошедшийся предел каждого вклада при данном α.')
print('Ошибки ниже 1e-12 kJ/mol показаны на уровне 1e-12; это предел отображения.')


## 6. Компромисс между реальным и обратным пространствами

На картах цвет кодирует $\log_{10}|U-U_{\rm ref}|$. По горизонтали — rcut прямой суммы, по вертикали — nmax обратной. Для отображения ошибки ниже $10^{-12}$ kJ/mol ограничены этим уровнем. Квадрат отмечает точку с ошибкой менее $10^{-3}$ kJ/mol.

При большем α можно уменьшить реальный cutoff, но потребуется больше обратных мод. Число обратных мод служит **оценкой объёма суммирования**, не точной моделью быстродействия. Векторизованный Python и PME имеют разные накладные расходы; этот опыт не является бенчмарком производительности.

In [ ]:
radii=np.array([.8,1.2,1.8,2.4,3.6,4.8])
nvalues=np.array([1,2,3,4,6,8])
map_alphas=[1.,2.,3.]
errors=[]
for alpha in map_alphas:
    errors.append(np.array([[abs(ewald(pos,q,L,alpha,rcut,int(n))[0].sum()-E_ref)
                            for rcut in radii] for n in nvalues]))
fig,axes=plt.subplots(1,3,figsize=(14,4),layout='constrained')
for ax,alpha,err in zip(axes,map_alphas,errors):
    im=ax.imshow(np.log10(np.maximum(err,1e-12)),origin='lower',aspect='auto',vmin=-8,vmax=2,cmap='viridis')
    ax.set_xticks(range(len(radii)),radii);ax.set_yticks(range(len(nvalues)),nvalues)
    ax.set(xlabel='rcut, nm',ylabel='nmax',title=f'α={alpha} nm⁻¹')
    rows,cols=np.where(err<1e-3);ax.scatter(cols,rows,marker='s',s=60,facecolors='none',edgecolors='white')
fig.colorbar(im,ax=axes,label='log10(|ΔU| / (kJ/mol))');plt.show()
print('Пример: число ненулевых обратных мод при nmax=4:',(2*4+1)**3-1)
print('В прямой сумме считаются пары с каждым включённым образом; используется множитель 1/2.')


## 7. PME в OpenMM: сравнение с независимым расчётом

PME использует то же разложение, но ускоряет обратную часть с помощью сетки:

**заряды → распределение на сетку → FFT → умножение на электростатическое ядро → обратное FFT → силы на атомах.**

Собственная NumPy-реализация суммирует обратные моды непосредственно: это обычный Эвальд, **не PME**. В OpenMM устанавливаем PME, те же заряды и ячейку, LJ-энергию ноль, без исключений. Обе модели используют проводящие внешние граничные условия.

`setEwaldErrorTolerance` задаёт внутренний критерий выбора параметров, а не строгую верхнюю границу ошибки энергии в kJ/mol. Мы отдельно измеряем ошибку энергии и RMS-ошибку декартовых компонент силы. Запрашиваем фактические α и размеры сетки из контекста. Результат на другом backend может немного отличаться.

Здесь энергии сопоставляются непосредственно при совпадающих соглашениях. В других пакетах могут применяться дополнительные постоянные сдвиги; их нужно учитывать, а силы от постоянного сдвига не меняются.

In [ ]:
def pme_context(tolerance=1e-4):
    system=mm.System();force=mm.NonbondedForce()
    force.setNonbondedMethod(mm.NonbondedForce.PME)
    force.setCutoffDistance(1.1*unit.nanometer)  # строго меньше L/2
    force.setEwaldErrorTolerance(tolerance)
    force.setUseDispersionCorrection(False)
    for charge in q:
        system.addParticle(40*unit.dalton)
        force.addParticle(charge*unit.elementary_charge,.3*unit.nanometer,0*unit.kilojoule_per_mole)
    system.addForce(force)
    system.setDefaultPeriodicBoxVectors(mm.Vec3(L,0,0)*unit.nanometer,
                                       mm.Vec3(0,L,0)*unit.nanometer,
                                       mm.Vec3(0,0,L)*unit.nanometer)
    integrator=mm.VerletIntegrator(.001*unit.picoseconds)  # нужен контексту; step не вызывается
    context=mm.Context(system,integrator,backend)
    context.setPositions(pos*unit.nanometer)
    return context,integrator,force

def read_energy_force(context):
    st=context.getState(getEnergy=True,getForces=True)
    return (st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole),
            st.getForces(asNumpy=True).value_in_unit(unit.kilojoule_per_mole/unit.nanometer))

tolerances=[1e-2,1e-3,1e-4,1e-5]
pme_rows=[]
for tol in tolerances:
    context,integrator,force=pme_context(tol)
    energy,forces=read_energy_force(context)
    alpha,nx,ny,nz=force.getPMEParametersInContext(context)
    errF=np.sqrt(np.mean((forces-F_ref)**2))
    pme_rows.append([tol,energy-E_ref,errF])
    print(f'tol={tol:g}  α={alpha:.4f}  grid={nx}×{ny}×{nz}  ΔU={energy-E_ref:+.6g}  RMS ΔF={errF:.6g}')
    del context,integrator
pme_rows=np.array(pme_rows)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].loglog(pme_rows[:,0],np.abs(pme_rows[:,1]),'o-');ax[0].set_ylabel('|ΔU|, kJ/mol')
ax[1].loglog(pme_rows[:,0],pme_rows[:,2],'o-');ax[1].set_ylabel('RMS ΔF, kJ/mol/nm')
for a in ax:a.set_xlabel('Ewald error tolerance');a.invert_xaxis()
plt.tight_layout();plt.show()
assert abs(pme_rows[-1,1])<.02
assert pme_rows[-1,2]<.02
print('Согласие с независимым Эвальдом проверено для этой конфигурации.')


## 8. Дополнительно: перенос зарядов относительно сетки

Точная периодическая энергия не меняется при одинаковом переносе всех зарядов. Но при конечной сетке PME остаётся небольшая зависимость от положения относительно её узлов. Здесь переносим систему на один шаг сетки вдоль x и сравниваем грубую и более точную настройки.

По оси x — доля шага **собственной** сетки каждого расчёта, поэтому абсолютные расстояния для двух кривых отличаются. По оси y — изменение относительно энергии до переноса в том же расчёте, а не ошибка относительно эталона. Радиус отсечения прямой суммы не меняется, как и все относительные расстояния.

In [ ]:
fractions=np.linspace(0,1,31)
fig,ax=plt.subplots()
for tol in [1e-2,1e-4]:
    context,integrator,force=pme_context(tol)
    alpha,nx,ny,nz=force.getPMEParametersInContext(context)
    energies=[]
    for fraction in fractions:
        shifted=(pos+np.array([fraction*L/nx,0,0]))%L
        context.setPositions(shifted*unit.nanometer)
        energies.append(read_energy_force(context)[0])
    energies=np.array(energies)
    ax.plot(fractions,energies-energies[0],'o-',ms=3,label=f'tol={tol}; nx={nx}')
    print('tol=',tol,'размах энергии при переносе:',np.ptp(energies),'kJ/mol')
    del context,integrator
ax.set(xlabel='Перенос / шаг сетки',ylabel='U(s)−U(0), kJ/mol');ax.legend();plt.show()
# Независимая проверка инвариантности обычного Эвальда.
shifted=(pos+np.array([.137,.219,.081]))%L
Eshift=ewald(shifted,q,L,ref_alpha,ref_rc,ref_n)[0].sum()
assert abs(Eshift-E_ref)<1e-7
print('Обычный Эвальд: изменение при переносе',Eshift-E_ref,'kJ/mol')


## Задания и вопросы

1. Увеличьте α: что произойдёт с шириной облака, real- и reciprocal-сходимостью?
2. Почему сумма меняется при α, если оставить слишком маленькие rcut и nmax? Изменилось ли физическое взаимодействие?
3. Почему self correction зависит от α, но не даёт силы?
4. Почему 1/k² не объясняет быструю сходимость всей обратной суммы в 3D?
5. По карте найдите два разных набора параметров с ошибкой менее 0.001 kJ/mol. Сравните число обратных мод.
6. Измените положение одного заряда, сохранив нейтральность, и повторите проверки. Является ли заданный допуск PME гарантией абсолютной ошибки энергии?
7. Что произойдёт, если удалить один отрицательный заряд? Почему учебная реализация отклоняет такой ввод?

<details><summary>Подсказки</summary>

1. Облако сужается; прямая сумма сходится быстрее, обратной нужно больше мод.
2. Изменилась ошибка усечения двух согласованных сумм, а не исходная периодическая задача.
3. Поправка зависит только от зарядов и параметра разбиения; её градиент по координатам нулевой.
4. Число мод в оболочке растёт, а структура зарядов даёт множитель |S|²; экспонента подавляет хвост.
5. Для куба индексов число мод (2nmax+1)³−1; число операций и реальное время не тождественны.
6. Это параметр алгоритмического выбора точности; ошибки нужно измерять для интересующей системы.
7. Появится ненулевой средний заряд. Нужна явно заданная модель компенсации и соответствующие поправки; код их не реализует.
</details>

## Источники и ограничения

- [OpenMM: NonbondedForce и параметры PME](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.NonbondedForce.html).
- [OpenMM: Standard Forces, кулоновское взаимодействие и PME](https://docs.openmm.org/latest/userguide/theory/02_standard_forces.html).
- [GROMACS: разложение Эвальда и PME](https://manual.gromacs.org/current/reference-manual/functions/long-range-electrostatics.html).
- [Sarkas: объяснение particle–particle / particle–mesh](https://murillo-group.github.io/sarkas/theory/PPPM.html).

Реализация NumPy рассчитана на несколько фиксированных зарядов в кубе, нейтральность и проводящие граничные условия. Она не поддерживает триклинные ячейки, исключения, поверхности, компенсирующий фон и эффективные списки соседей. Прямое суммирование несглаженного кулоновского ряда намеренно не используется как эталон: условная сходимость и порядок суммирования требуют отдельного рассмотрения.